# The Directional Derivative

Wiki reference for [the directional derivative](https://ml-viz-ruby.vercel.app/wiki/directional-derivative).

**The idea in one sentence.** The rate of change of $f$ along a unit direction $\hat u$ is
$D_{\hat u} f = \nabla f \cdot \hat u$, which is **maximised along the gradient** (rate
$= \|\nabla f\|$), **minimised opposite it** (steepest descent), and **zero perpendicular** to
it (moving along a level set) — the geometric fact that makes gradient descent work.

We implement the directional derivative two ways from scratch, **validate that the limit and
dot-product forms agree and that the gradient is the steepest direction**, then cover the
gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — The wiki's worked example

Function: $f(x,y) = x^2 + 3xy$, evaluated at $\mathbf{x} = (1, 2)$.

Gradient $\nabla f = [2x + 3y,\; 3x]$, so $\nabla f(1,2) = [8, 3]$.

The directional derivative toward $\mathbf{v} = (3, 4)$ should come out to $7.2$.

In [ ]:
def f(xy):
    x, y = xy
    return x**2 + 3*x*y

def grad_f(xy):
    x, y = xy
    return np.array([2*x + 3*y, 3*x])

p = np.array([1.0, 2.0])
g = grad_f(p)                       # [8., 3.]

v = np.array([3.0, 4.0])
u = v / np.linalg.norm(v)           # unit vector [0.6, 0.8]
Du = g @ u                          # directional derivative

print(f"gradient        ∇f = {g}")
print(f"unit direction  û  = {u}")
print(f"D_û f            = {Du:.3f}")          # 7.200
print(f"steepest rate   = {np.linalg.norm(g):.3f}")  # 8.544 (>= 7.200)

## 2 — Limit definition vs. dot-product formula

The directional derivative is *defined* as a limit:

$$D_{\hat{\mathbf{u}}} f(\mathbf{x}) = \lim_{h \to 0} \frac{f(\mathbf{x} + h\,\hat{\mathbf{u}}) - f(\mathbf{x})}{h}$$

and (where $f$ is differentiable) *equals* $\nabla f \cdot \hat{\mathbf{u}}$. A finite-difference
approximation of the limit should match the dot product to several digits.

In [ ]:
def directional_derivative_limit(f, x, u, h=1e-6):
    """Forward finite difference of the limit definition. u must be a unit vector."""
    u = u / np.linalg.norm(u)
    return (f(x + h * u) - f(x)) / h

def directional_derivative_dot(grad, x, u):
    """Dot-product formula: ∇f · û."""
    u = u / np.linalg.norm(u)
    return grad(x) @ u

approx = directional_derivative_limit(f, p, v)
exact  = directional_derivative_dot(grad_f, p, v)
print(f"finite-difference: {approx:.6f}")
print(f"∇f · û           : {exact:.6f}")
assert np.isclose(approx, exact, atol=1e-4)
print("✓ limit definition and dot-product formula agree")

### Validate: the limit definition equals the dot-product formula

The directional derivative can be computed as a finite-difference limit
$\lim_{h\to0}\frac{f(x+h\hat u)-f(x)}{h}$ or as $\nabla f \cdot \hat u$ — and they must agree.
We confirm the two forms match, and that no direction beats the gradient's magnitude.

In [ ]:
print(f'finite-difference: {approx:.6f}   dot-product: {exact:.6f}')
assert np.isclose(approx, exact, atol=1e-4), 'the limit definition matches grad . u_hat'
assert np.linalg.norm(g) >= exact - 1e-9, 'no direction exceeds the gradient magnitude (steepest ascent)'
print('\n✅ D_u f = grad f . u_hat — the projection of the gradient onto the direction')

## 3 — Sweeping the direction: steepest ascent emerges

Rotate $\hat{\mathbf{u}}$ through a full circle and plot $D_{\hat{\mathbf{u}}} f$ against the angle.
The curve is $\|\nabla f\|\cos\theta$: it peaks at $+\|\nabla f\|$ when $\hat{\mathbf{u}}$ aligns
with the gradient and bottoms out at $-\|\nabla f\|$ in the opposite direction.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 361)
dirs = np.stack([np.cos(angles), np.sin(angles)], axis=1)
Dvals = dirs @ g                      # ∇f · û for each unit direction

grad_angle = np.arctan2(g[1], g[0])
grad_norm = np.linalg.norm(g)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.degrees(angles), Dvals, color='#6366f1')
ax.axhline(grad_norm,  ls='--', color='#2dd4bf', label=f'+‖∇f‖ = {grad_norm:.2f}')
ax.axhline(-grad_norm, ls='--', color='#fb7185', label=f'-‖∇f‖ = {-grad_norm:.2f}')
ax.axhline(0, color='#555', lw=1)
ax.axvline(np.degrees(grad_angle), color='#2dd4bf', lw=1, alpha=0.6)
ax.set_xlabel('direction angle θ (degrees)')
ax.set_ylabel('directional derivative  D_û f')
ax.set_title('Directional derivative vs. direction  —  peaks along ∇f')
ax.legend(facecolor='#1a1d27', edgecolor='#444')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best = np.degrees(angles[np.argmax(Dvals)])
print(f"max D_û f at θ = {best:.1f}°,  gradient at θ = {np.degrees(grad_angle):.1f}°")

### Validate: the gradient is the steepest direction

Sweeping over all unit directions, the directional derivative ranges exactly between
$+\|\nabla f\|$ (along the gradient) and $-\|\nabla f\|$ (opposite it). We confirm the extremes
of the swept curve equal $\pm\|\nabla f\|$.

In [ ]:
print(f'max D_u f over all directions = {Dvals.max():.3f},  ||grad f|| = {grad_norm:.3f}')
print(f'min D_u f over all directions = {Dvals.min():.3f}, -||grad f|| = {-grad_norm:.3f}')
assert np.isclose(Dvals.max(), grad_norm, atol=1e-2), 'steepest ascent rate = +||grad f|| (along the gradient)'
assert np.isclose(Dvals.min(), -grad_norm, atol=1e-2), 'steepest descent rate = -||grad f|| (opposite)'
print('\n✅ the gradient points in the steepest-ascent direction; its magnitude is the max rate')

## 4 — Geometry on a contour plot

The gradient is **perpendicular to the level set** through the point, because moving along a
contour leaves $f$ unchanged ($D_{\hat{\mathbf{u}}} f = 0$). We overlay the gradient (steepest
ascent) and our chosen direction $(3,4)$ on the contours of $f$.

In [ ]:
xs = np.linspace(-1, 3, 200)
ys = np.linspace(0, 4, 200)
X, Y = np.meshgrid(xs, ys)
Z = X**2 + 3 * X * Y

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contour(X, Y, Z, levels=18, cmap='viridis', alpha=0.8)
ax.clabel(cs, inline=True, fontsize=7)

# gradient direction (steepest ascent) and the chosen direction, scaled for display
ax.quiver(*p, *(g / grad_norm), color='#2dd4bf', scale=6, width=0.012,
          label='∇f / ‖∇f‖  (steepest ascent)')
ax.quiver(*p, *u, color='#fb7185', scale=6, width=0.012,
          label='û  (toward (3,4))')
ax.plot(*p, 'o', color='#eee', ms=6)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Contours of f(x,y) = x² + 3xy  at (1, 2)')
ax.legend(facecolor='#1a1d27', edgecolor='#444', loc='upper left')
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **non-unit direction** | must normalize $\hat u$ or the "rate" is mis-scaled |
| **gradient is local** | steepest-descent is only the best *instantaneous* direction |
| **ill-conditioning** | steepest descent zig-zags on elongated valleys — see momentum |
| **non-differentiable points** | the directional derivative may not exist / be direction-dependent |
| **max vs min** | $+\|\nabla f\|$ is ascent; descent needs the minus sign |

Demo: steepest descent is $-\nabla f$; perpendicular to the gradient is a level set (zero rate).

In [ ]:
# The geometric fact gradient descent is built on: the steepest-DESCENT direction is exactly
# -grad f (rate -||grad f||), while moving PERPENDICULAR to the gradient changes f at rate ZERO
# — that perpendicular direction is the tangent to the level set (contour line). We confirm both.
descent = -g / np.linalg.norm(g)
rate_descent = g @ descent
perp = np.array([-g[1], g[0]]); perp = perp / np.linalg.norm(perp)   # rotate gradient 90 degrees
rate_perp = g @ perp
print(f'rate along -grad (steepest descent): {rate_descent:.3f}  (= -||grad|| = {-np.linalg.norm(g):.3f})')
print(f'rate perpendicular to grad         : {rate_perp:.3e}  (~0 -> along a level set)')
assert np.isclose(rate_descent, -np.linalg.norm(g)), 'steepest descent has rate -||grad f||'
assert abs(rate_perp) < 1e-9, 'moving perpendicular to the gradient changes f at zero rate (level set)'
print('\nGradient descent steps along -grad (fastest decrease); level sets are perpendicular to the gradient.')

## ✏️ Your turn

**Exercise.** Implement `steepest_descent_direction(grad, x)` returning the **unit vector** that
*minimizes* the directional derivative at `x` (the direction a gradient-descent step would take),
and `max_decrease_rate(grad, x)` returning the value of $D_{\hat{\mathbf{u}}} f$ along it.

Recall: steepest descent is $-\nabla f / \|\nabla f\|$ and the most negative directional
derivative is $-\|\nabla f\|$.

In [ ]:
def steepest_descent_direction(grad, x):
    g = grad(x)
    # TODO(you): return the unit vector pointing in the steepest-descent direction
    return ...

def max_decrease_rate(grad, x):
    g = grad(x)
    # TODO(you): return the directional derivative along steepest descent (a negative scalar)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
d = steepest_descent_direction(grad_f, p)
rate = max_decrease_rate(grad_f, p)

assert np.isclose(np.linalg.norm(d), 1.0), "direction must be a unit vector"
assert np.allclose(d, -g / np.linalg.norm(g)), "should point opposite the gradient"
assert np.isclose(rate, -np.linalg.norm(g)), "rate should equal -‖∇f‖"
assert np.isclose(grad_f(p) @ d, rate), "rate must equal ∇f · d"
print("✓ all checks passed")

<details>
<summary>Solution</summary>

```python
def steepest_descent_direction(grad, x):
    g = grad(x)
    return -g / np.linalg.norm(g)

def max_decrease_rate(grad, x):
    g = grad(x)
    return -np.linalg.norm(g)
```

The directional derivative $D_{\hat{\mathbf{u}}} f = \|\nabla f\|\cos\theta$ is minimized at
$\theta = \pi$, i.e. when $\hat{\mathbf{u}}$ points opposite the gradient. Its value there is
$-\|\nabla f\|$ — the fastest possible rate of decrease, which is exactly the step gradient
descent takes.

</details>

## Key takeaways

- **$D_{\hat u} f = \nabla f \cdot \hat u$:** the limit and dot-product forms agree (verified).
- **Gradient = steepest ascent:** the max rate over all directions is $\|\nabla f\|$
  (verified).
- **Steepest descent = $-\nabla f$:** rate $-\|\nabla f\|$ — what gradient descent follows
  (demo).
- **Perpendicular = zero change:** the gradient is normal to the level sets (demo).